## Open-Sora on Google Colab with Gradio Interface and Ngrok Tunnel

This notebook allows you to run Open-Sora to generate videos from text prompts using Google Colab's GPU resources. It provides a Gradio web interface to interact with the model, and uses Ngrok to create a public URL for accessing the interface from your local browser on Windows (or any other OS).

### 1. Setup Environment and Install Dependencies

In [ ]:
!pip install ninja colossalai mmengine gradio pyngrok

# Install PyTorch (ensure compatibility with Colab's CUDA version - Colab usually has recent versions)
# Check Colab's PyTorch and CUDA version if issues arise, might need specific versions like cu118 or cu121
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install xformers (ensure compatibility with PyTorch and CUDA)
!pip install xformers --index-url https://download.pytorch.org/whl/cu121

# Apex (optional, but often used in HPC AI projects, can be tricky on Colab sometimes)
%cd /content
!git clone -b 23.05-devel https://github.com/NVIDIA/apex
%cd /content/apex
!pip install -v --disable-pip-version-check --no-cache-dir --global-option="--cpp_ext" --global-option="--cuda_ext" ./

# Flash Attention (also optional, but good for performance if compatible)
%cd /content
!git clone https://github.com/Dao-AILab/flash-attention
%cd /content/flash-attention
!pip install -v -e .

# Clone Open-Sora repository
%cd /content
!git clone https://github.com/hpcaitech/Open-Sora
# !git clone -b dev https://github.com/camenduru/Open-Sora # Alternative fork, the original one is hpcaitech
%cd /content/Open-Sora
!pip install -v -e .

### 2. Download Pre-trained Models

We'll download a smaller model for quicker testing first. You can add more models later.

In [ ]:
!apt -y install -qq aria2

# Download the 16x256x256 model (smaller, faster for testing)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/hpcai-tech/Open-Sora/resolve/main/OpenSora-v1-16x256x256.pth -d /content/Open-Sora/models -o OpenSora-v1-16x256x256.pth

# (Optional) Download a higher quality model (e.g., 16x512x512) - uncomment if you have Colab Pro and more time
# !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/hpcai-tech/Open-Sora/resolve/main/OpenSora-v1-HQ-16x512x512.pth -d /content/Open-Sora/models -o OpenSora-v1-HQ-16x512x512.pth

# Download T5 text encoder model
!git clone https://huggingface.co/DeepFloyd/t5-v1_1-xxl /content/Open-Sora/pretrained_models/t5_ckpts/t5-v1_1-xxl

### 3. Define Inference Function and Gradio Interface

In [ ]:
import gradio as gr
import subprocess
import os
import re
import time

os.chdir('/content/Open-Sora') # Ensure we are in the correct directory

VIDEO_OUTPUT_DIR = '/content/Open-Sora/samples'
if not os.path.exists(VIDEO_OUTPUT_DIR):
    os.makedirs(VIDEO_OUTPUT_DIR)

MODEL_OPTIONS = {
    "16x256x256 (Fastest)": {
        "config": "configs/opensora/inference/16x256x256.py",
        "ckpt_path": "/content/Open-Sora/models/OpenSora-v1-16x256x256.pth"
    },
    # Add more models here if you download them, e.g.:
    # "16x512x512 (HQ)": {
    #     "config": "configs/opensora/inference/16x512x512.py",
    #     "ckpt_path": "/content/Open-Sora/models/OpenSora-v1-HQ-16x512x512.pth"
    # }
}

def generate_video(prompt, model_choice):
    if not prompt:
        return None, "Error: Prompt cannot be empty."
    
    selected_model = MODEL_OPTIONS.get(model_choice)
    if not selected_model:
        return None, f"Error: Model {model_choice} not found."

    config_path = selected_model["config"]
    ckpt_path = selected_model["ckpt_path"]

    if not os.path.exists(ckpt_path):
        return None, f"Error: Model checkpoint {ckpt_path} not found. Please ensure it's downloaded."
    
    # Clean up old samples from the specific config to avoid confusion if script doesn't overwrite well
    # This is a simple cleanup, more robust cleanup might be needed depending on how inference.py saves files
    # sample_subdir = os.path.join(VIDEO_OUTPUT_DIR, os.path.splitext(os.path.basename(config_path))[0])
    # if os.path.exists(sample_subdir):
    #     for f in os.listdir(sample_subdir):
    #         if f.endswith('.mp4'): os.remove(os.path.join(sample_subdir, f))

    # Construct the command. Using a unique save directory based on timestamp to avoid clashes.
    # The inference script might save files in a nested structure based on the config name.
    # We need to find the most recently created mp4 file in the expected output directory.
    
    # The script saves under samples/<config_name>/<prompt_slug>/video.mp4
    # Let's make a simple slug from the prompt for the output directory
    prompt_slug = "".join(filter(str.isalnum, prompt.lower().replace(' ', '_')))[:50]
    # The inference script itself will create a subdirectory based on the config name inside 'samples'
    # and then another one for the prompt.

    command = [
        'torchrun',
        '--standalone',
        '--nproc_per_node=1',
        'scripts/inference.py',
        config_path,
        f'--ckpt-path={ckpt_path}',
        f'--prompt="{prompt}"',
        f'--save-dir={VIDEO_OUTPUT_DIR}' # The script seems to handle subdirs itself
    ]
    
    print(f"Running command: {' '.join(command)}")
    
    try:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, cwd='/content/Open-Sora')
        stdout, stderr = process.communicate(timeout=600) # 10 minutes timeout
        
        print("Stdout:")
        print(stdout)
        print("Stderr:")
        print(stderr)

        if process.returncode != 0:
            return None, f"Error during video generation. Return code: {process.returncode}\nStderr: {stderr}"
        
        # Try to find the generated video file
        # The script saves to samples/<config_name_without_py>/<prompt_slug_or_text>/sample_0/video.mp4 (or similar)
        # This part is tricky as the exact output path can vary based on the script's internal logic.
        # A robust way is to search for the newest .mp4 file in the VIDEO_OUTPUT_DIR and its subdirectories.
        
        latest_video = None
        latest_time = 0

        # Config name part of the path
        config_name_part = os.path.splitext(os.path.basename(config_path))[0]
        # The script might further create subdirectories based on the prompt or a timestamp
        # For simplicity, let's scan the base config output directory
        search_dir = os.path.join(VIDEO_OUTPUT_DIR, config_name_part)
        print(f"Searching for video in and under: {search_dir}")

        for root, dirs, files in os.walk(search_dir):
            for file in files:
                if file.endswith(".mp4"):
                    file_path = os.path.join(root, file)
                    file_time = os.path.getmtime(file_path)
                    if file_time > latest_time:
                        latest_time = file_time
                        latest_video = file_path
        
        if latest_video:
            print(f"Found video: {latest_video}")
            return latest_video, f"Video generated successfully: {os.path.basename(latest_video)}"
        else:
            return None, "Error: Video file not found after generation. Check logs for output path."

    except subprocess.TimeoutExpired:
        process.kill()
        stdout, stderr = process.communicate()
        print("Stdout (Timeout):")
        print(stdout)
        print("Stderr (Timeout):")
        print(stderr)
        return None, "Error: Video generation timed out after 10 minutes."
    except Exception as e:
        return None, f"An unexpected error occurred: {str(e)}"

with gr.Blocks() as demo:
    gr.Markdown("## Open-Sora Text-to-Video Generator")
    with gr.Row():
        prompt_input = gr.Textbox(label="Enter your prompt", placeholder="A cat wearing a hat...")
    with gr.Row():
        model_dropdown = gr.Dropdown(label="Select Model", choices=list(MODEL_OPTIONS.keys()), value=list(MODEL_OPTIONS.keys())[0])
    with gr.Row():
        generate_button = gr.Button("Generate Video")
    with gr.Row():
        video_output = gr.Video(label="Generated Video")
    with gr.Row():
        status_output = gr.Textbox(label="Status")

    generate_button.click(
        generate_video,
        inputs=[prompt_input, model_dropdown],
        outputs=[video_output, status_output]
    )

print("Gradio interface defined.")

### 4. Setup Ngrok and Launch the Gradio App

You'll need an Ngrok authtoken. 
1. Go to [https://dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)
2. Copy your authtoken.
3. Paste it into the `ngrok_authtoken` variable cell below.

In [ ]:
# PASTE YOUR NGROK AUTHTOKEN HERE
ngrok_authtoken = "YOUR_NGROK_AUTHTOKEN" # Replace with your actual token

In [ ]:
from pyngrok import ngrok, conf
import gc

if ngrok_authtoken == "YOUR_NGROK_AUTHTOKEN" or not ngrok_authtoken:
    print("ERROR: Please set your Ngrok authtoken in the cell above!")
else:
    try:
        conf.get_default().auth_token = ngrok_authtoken
        # Kill any existing ngrok tunnels
        for tunnel in ngrok.get_tunnels():
            ngrok.disconnect(tunnel.public_url)
            ngrok.kill()
            print(f"Killed existing tunnel: {tunnel.public_url}")
        # Launch the Gradio app and create a tunnel
        # Ensure CUDA cache is cleared if you had previous runs or OOM errors
        # torch.cuda.empty_cache()
        # gc.collect()
        public_url = ngrok.connect(7860) # Gradio default port is 7860
        print(f"Gradio App is running at: {public_url}")
        print("Please open this URL in your browser on Windows (or any other device).")
        demo.launch(share=False) # share=False because ngrok is handling it
    except Exception as e:
        print(f"An error occurred with Ngrok or Gradio launch: {e}")
        print("Make sure your ngrok authtoken is correct and that you haven't exceeded ngrok's concurrent tunnel limits for your account type.")
        print("If you see ' ricevuto segnale di interruzione (signal received)', it might be Colab resource limits or an internal Gradio/Ngrok issue.")

### Important Notes:
*   **Ngrok URL:** Once the last cell runs successfully, it will print an Ngrok URL (e.g., `https://xxxx-xx-xx-xx-xx.ngrok.io`). Open this URL in your browser on Windows to access the Gradio interface.
*   **Colab Runtime:** The Colab notebook must remain running for the Gradio app and Ngrok tunnel to stay active. If the Colab runtime disconnects or times out, you'll lose access.
*   **Model Download Time:** The first time you run this, downloading models will take a while.
*   **Generation Time:** Video generation can be slow, especially for longer prompts or if using higher-quality models (once added). Be patient.
*   **Resource Limits:** Colab (especially the free tier) has resource limits (GPU time, RAM, disk space). If you encounter errors like 'Out of Memory' (OOM) or disconnections, you might be hitting these limits. Colab Pro/Pro+ offers more resources.
*   **Error Handling:** The `generate_video` function has basic error handling and a timeout. Check the Colab output cells for detailed logs if generation fails.
*   **Output Files:** Generated videos are saved in `/content/Open-Sora/samples/...` within the Colab environment. The Gradio interface will display the video, but if you need the raw file, you can download it from Colab's file browser.
*   **Ngrok Authtoken:** Ensure you've correctly set your Ngrok authtoken in the designated cell. Without it, Ngrok won't work.